Log

In [ ]:
%run Utils_Log

In [ ]:
setup_log("Ingesta_2026")

In [7]:
# ── Instalación ────────────────────────────────────────────────────────────────
#%pip install requests beautifulsoup4 pandas -q
 
import requests
import pandas as pd
import json
import math
import time
import os
from io import StringIO
from bs4 import BeautifulSoup
from datetime import date
 
# ── Configuración ──────────────────────────────────────────────────────────────
BASE_URL   = "https://www.pescadegalicia.gal"
PAGE_URL   = f"{BASE_URL}/informe-notasventa/agregation"
TODAY      = date.today()
PAGE_SIZE  = 10000
OUTPUT_PATH = "Files/Bronze/Ventas/anio=2026/pescafresca_2026.csv"  # ruta Lakehouse
 
HEADERS_BASE = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Accept-Language": "es-ES,es;q=0.9",
    "Referer": PAGE_URL,
    "Origin": BASE_URL,
    "X-Requested-With": "XMLHttpRequest",
    "Sec-Fetch-Dest": "empty",
    "Sec-Fetch-Mode": "cors",
    "Sec-Fetch-Site": "same-origin",
}

 
# ── Payload base ───────────────────────────────────────────────────────────────
# Columnas a agrupar (fecha + dimensiones, sin ejercicio/año)
COLUMNAS_GROUPS = [
    {"updated": True,  "selected": True,  "agregated": True,  "codigo": "fecha",                  "path": ""},
    {"updated": False, "selected": False, "agregated": False, "codigo": "diaSemana",              "path": ""},
    {"updated": True,  "selected": True,  "agregated": True,  "codigo": "provincia",              "path": ""},
    {"updated": True,  "selected": True,  "agregated": True,  "codigo": "zonaAdministrativa",     "path": ""},
    {"updated": True,  "selected": True,  "agregated": True,  "codigo": "lonjaAgrupacionView",    "path": ""},
    {"updated": True,  "selected": True,  "agregated": True,  "codigo": "grupoBiologico",         "path": ""},
    {"updated": True,  "selected": True,  "agregated": True,  "codigo": "especie",                "path": ""},
    {"updated": False, "selected": False, "agregated": False, "codigo": "ejercicio",              "path": ""},
    {"updated": False, "selected": False, "agregated": False, "codigo": "mes",                    "path": ""},
]
 
COLUMNAS_INDEXES = [
    {"updated": True,  "selected": True,  "agregated": True,  "codigo": "cantidadsum",            "path": ""},
    {"updated": True,  "selected": True,  "agregated": True,  "codigo": "importesum",             "path": ""},
    {"updated": False, "selected": False, "agregated": False, "codigo": "precioMedioany",         "path": ""},
    {"updated": False, "selected": False, "agregated": False, "codigo": "precioMinimomin",        "path": ""},
    {"updated": False, "selected": False, "agregated": False, "codigo": "precioMaximomax",        "path": ""},
    {"updated": False, "selected": False, "agregated": False, "codigo": "fechamax",               "path": ""},
    {"updated": False, "selected": False, "agregated": False, "codigo": "provinciacdt",           "path": ""},
    {"updated": False, "selected": False, "agregated": False, "codigo": "zonaAdministrativacdt",  "path": ""},
    {"updated": False, "selected": False, "agregated": False, "codigo": "lonjaAgrupacionViewcdt", "path": ""},
    {"updated": False, "selected": False, "agregated": False, "codigo": "grupoBiologicocdt",      "path": ""},
    {"updated": False, "selected": False, "agregated": False, "codigo": "especiecdt",             "path": ""},
    {"updated": False, "selected": False, "agregated": False, "codigo": "ejerciciocdt",           "path": ""},
    {"updated": False, "selected": False, "agregated": False, "codigo": "mesAgregadocdt",         "path": ""},
    {"updated": False, "selected": False, "agregated": False, "codigo": "fechacdt",               "path": ""},
]
 
# Filtros: año en curso + todas las zonas + provincias + especies sin "Resto"
FILTROS = [
    {
        "tipo": "TYPE_FECHA",
        "codigo": "fecha",
        "path": "fecha",
        "isDatetime": False,
        "isTimeOnly": False,
        "valorDesde": f"01/01/{TODAY.year}",
        "valorHasta": TODAY.strftime("%d/%m/%Y"),
        "texto": f"01/01/{TODAY.year} - {TODAY.strftime('%d/%m/%Y')}",
    },
    {
        "tipo": "TYPE_TREE_MULTIPLE",
        "codigo": "lonjaAgrupacionViewZonaAdministrativa",
        "path": "lonjaAgrupacionViewZonaAdministrativa.id",
        "treeHierarchy": ["lonjaAgrupacionViewZonaAdministrativa", "zonaAdministrativa"],
        "treeNodeFinal": True,
        "valor": ["0_1","0_2","0_3","0_4","0_5","0_6","0_7","0_8","0_9"],
        "texto": "Todas las zonas",
    },
    {
        "tipo": "TYPE_TREE_MULTIPLE",
        "codigo": "lonjaAgrupacionView",
        "path": "lonjaAgrupacionView.id",
        "treeHierarchy": ["lonjaAgrupacionView", "provincia"],
        "treeNodeFinal": True,
        "valor": ["0_15","0_27","0_36"],   # A Coruña, Lugo, Pontevedra
        "texto": "A Coruña, Lugo, Pontevedra",
    },
    {
        "tipo": "TYPE_TREE_MULTIPLE",
        "codigo": "especie",
        "path": "especie.id",
        "treeHierarchy": ["especie", "grupoBiologico"],
        "treeNodeFinal": True,
        "valor": ["0_1","0_2","0_3","0_4","0_5","0_6","0_7","0_8"],  # sin Resto das descargas
        "texto": "Algas, Bivalvos, Cefalópodos, Crustáceos, Equinodermos, Gasterópodos, Peixes, Poliquetos",
    },
]
 
 
# ── 1. Obtener CSRF token y cookies de sesión ─────────────────────────────────
def get_session():
    session = requests.Session()
    r = session.get(PAGE_URL, headers={"User-Agent": HEADERS_BASE["User-Agent"]}, timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    csrf = soup.find("meta", {"name": "_csrf"})
    csrf_token = csrf["content"] if csrf else None
    print(f"  CSRF token: {'OK' if csrf_token else 'NO ENCONTRADO'}")
    print(f"  Cookies:    {dict(session.cookies)}")
    return session, csrf_token
 
 
# ── 2. Hacer una petición de una página ───────────────────────────────────────
def fetch_page(session, csrf_token, page: int, total: int = 0):
    payload = {
        "totalRegistros": str(total),
        "paginaActual": page,
        "tamanhoPagina": PAGE_SIZE,
        "ordenacionList": [],
        "columnasAgregateGroups":  COLUMNAS_GROUPS,
        "columnasAgregateIndexes": COLUMNAS_INDEXES,
        "headerOrder": [],
        "mailData": {},
        "listaFiltros": FILTROS,
    }
    headers = {
        **HEADERS_BASE,
        "Content-Type": "application/json",
        "X-CSRF-TOKEN": csrf_token,
    }
    r = session.post(
        f"{PAGE_URL}?_referer=%252Finforme-notasventa%252Fagregation",
        headers=headers,
        json=payload,
        timeout=60,
    )
    r.raise_for_status()
    return r.text
 
 
# ── 3. Parsear HTML de respuesta a DataFrame ──────────────────────────────────
def parse_html(html: str) -> pd.DataFrame:
    tables = pd.read_html(StringIO(html))
    if not tables:
        return pd.DataFrame()
    df = max(tables, key=len)
    mask = df.iloc[:, 0].astype(str).str.lower().isin(["totais:", "totales:"])
    return df[~mask]
 
 
# ── 4. Extraer total de registros del HTML ────────────────────────────────────
def extract_total(html: str) -> int:
    soup = BeautifulSoup(html, "html.parser")
    # Buscar elemento con el total (suele estar en un span con id o clase específica)
    for selector in ["#totalRegistros", ".total-registros", "[id*='total']"]:
        el = soup.select_one(selector)
        if el:
            try:
                return int(el.text.strip().replace(".", "").replace(",", ""))
            except ValueError:
                pass
    # Fallback: usar el número de filas de la primera página para estimar
    return 0
 
 
# ── 5. Orquestador principal ──────────────────────────────────────────────────
print("=" * 60)
print(f"  Scraper año en curso ({TODAY.year}) — Fabric")
print("=" * 60)
 
print("\n  Obteniendo sesión y CSRF token...")
session, csrf_token = get_session()
 
if not csrf_token:
    raise ValueError("No se pudo obtener el CSRF token. Revisa la conexión.")
 
# Primera página para conocer el total de registros
print("\n  Página 1 (sondeo)...", end=" ", flush=True)
html_p1   = fetch_page(session, csrf_token, page=1, total=0)
df_p1     = parse_html(html_p1)
total_reg = extract_total(html_p1)
print(f"{len(df_p1)} filas  |  total registros: {total_reg or '?'}")

all_frames = [df_p1]
 
# Si tenemos el total, calculamos páginas; si no, paginamos hasta que no haya datos
if total_reg > 0:
    total_pages = math.ceil(total_reg / PAGE_SIZE)
else:
    total_pages = 9999  # paginamos hasta que la página esté vacía
 
for page in range(2, total_pages + 1):
    print(f"  Página {page}...", end=" ", flush=True)
    html   = fetch_page(session, csrf_token, page=page, total=total_reg)
    df_pag = parse_html(html)
    print(f"{len(df_pag)} filas")
    if df_pag.empty:
        break
    all_frames.append(df_pag)
    time.sleep(0.3)
    
# ── 6. Unir todas las páginas ──
df = pd.concat(all_frames, ignore_index=True)

# ── 7. Guardar CSV RAW ──
ruta_csv_absoluta = f"/lakehouse/default/{OUTPUT_PATH}"
os.makedirs(os.path.dirname(ruta_csv_absoluta), exist_ok=True)

df.to_csv(ruta_csv_absoluta, index=False, sep=";", encoding="utf-8-sig")
print(f"  Guardado CSV: {ruta_csv_absoluta}")
log("CSV de 2026 creado con exito")
df.head(5)

StatementMeta(, be5fa87f-c0d9-457f-8431-23879cd68240, 47, Finished, Available, Finished, False)


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
  Scraper año en curso (2026) — Fabric

  Obteniendo sesión y CSRF token...
  CSRF token: OK
  Cookies:    {'JSESSIONID': '68D0B0EF646378EED41CCDB22CD8DBB7', 'BIGipServer~ALTIA~PRO-PescaGalicia_TOMCAT10priv-HTTP': 'rd21o00000000000000000000ffff0a0a1ec5o8080'}

  Página 1 (sondeo)... 10000 filas  |  total registros: ?
  Página 2... 10000 filas
  Página 3... 10000 filas
  Página 4... 10000 filas
  Página 5... 10000 filas
  Página 6... 10000 filas
  Página 7... 10000 filas
  Página 8... 2473 filas
  Página 9... 0 filas
  Guardado CSV: /lakehouse/default/Files/Bronze/Ventas/anio=2026/pescafresca_2026.csv


,Data de venda,Provincia,Zona de produción,Lonxa,Grupo biolóxico,Especie,Cantidade (kg),Importe (€),Unnamed: 8
0,26/02/2026,Pontevedra,Zona III - Arousa,A Illa de Arousa,Algas,Golfo,"2.556,00","2.044,80",NaN
1,03/03/2026,Pontevedra,Zona III - Arousa,A Illa de Arousa,Algas,Golfo,"2.385,00","1.908,00",NaN
2,04/03/2026,Pontevedra,Zona III - Arousa,A Illa de Arousa,Algas,Golfo,"2.368,00","1.914,40",NaN
3,10/03/2026,Pontevedra,Zona III - Arousa,A Illa de Arousa,Algas,Golfo,"2.737,00","2.189,60",NaN
4,11/03/2026,Pontevedra,Zona III - Arousa,O Grove,Algas,Golfo,20000,16000,NaN


In [ ]:
#mssparkutils.session.stop()